<a href="https://colab.research.google.com/github/CathalD/BlueCarbon_Workflow_V1.0/blob/main/GlobalCoreCovariate_Extraction%2BSimilarityAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CathalD/NorthStarProject_CoastalBlueCarbonMMRV/blob/main/CoastalBlueCarbon_GlobalCoreCovariate_Extraction.ipynb)

# Global Soil Core Covariate Extraction
## Harmonized WOSIS + CanPeat + Janousek → GEE Remote Sensing Covariates

Extracts remote sensing covariate bands at all harmonized soil profile locations from
`combined_profiles_filtered.csv` (output of `05_filter_profiles.R`) using the
**Google Earth Engine Python API** (`reduceRegions`).

### Two-stage filtering
| Stage | Where | Filter |
|-------|-------|--------|
| **Geographic + depth** | R script 05 | lat ≥ 30°N, lon -170 to +60°E, depth ≥ 30 cm |
| **Climate analog** | This notebook (Cells 3c–5b) | MAT within ±2°C and MAP within ±30% of Charlie's Place KBA |

> **Efficiency note:** TerraClimate (4 km resolution) is extracted first at all ~15,000–16,000
> geo+depth filtered points (Cell 5). The climate analog filter is applied immediately (Cell 5b),
> reducing the point set to ~8,000–13,000 **before** the computationally expensive Sentinel-2
> extraction runs (Cell 5d). This avoids the `EEException: User memory limit exceeded` error
> that occurs when S2 is run on the full ~15,000-point set.

### Cluster-then-Match embedding similarity (Cell 6)
Instead of exporting all 64 raw embedding bands, Cell 6 runs a **Cluster-then-Match** analysis:
1. K-means clusters all AOI pixels into k=5 land-cover types (GEE server-side, `wekaKMeans`)
2. The medoid vector (pixel closest to cluster centroid) is extracted for each cluster
3. Cosine similarity between each global core's embedding and all 5 medoid vectors is computed in Python (numpy)
4. The resulting `embedding_sim_to_cluster_N` columns serve as transfer-learning weights in R

This produces compact, interpretable outputs instead of a ~10k × 64 raw embedding matrix.

### Outputs
| File | Content |
|------|---------|
| `CorePoints_Covariates_CSV.csv` | Topo + SAR + S2 optical + TerraClimate at climate-filtered points |
| `CorePoints_EmbeddingSimilarity_CSV.csv` | Cosine similarity to each of the k=5 AOI land-cover clusters + `embedding_max_sim` weight |
| `AOI_Medoids.csv` | k × 64 medoid embedding vectors (one representative per AOI cluster) |
| `AOI_KMeans_Clusters.tif` | 2-band GeoTIFF: Band 1 = cluster label (0–4), Band 2 = cosine sim to nearest medoid; 30 m, EPSG:4326 |
| `CorePoints_Climate_Filtered_OUT.csv` | Audit — profiles removed by climate filter |

---
### Prerequisites
1. Run `05_filter_profiles.R` first to produce `combined_profiles_filtered.csv`
2. Upload `combined_profiles_filtered.csv` to this Colab session (Cell 2)
3. Run cells in order: Auth → Load → Stacks → Climate ref → TerraClimate → Climate filter → Rebuild → Topo/S1/S2 → Cluster-then-Match → Export

In [1]:
# ============================================================================
# 1. AUTHENTICATION + SETUP
# ============================================================================
import ee

# Authenticate with your Google account (browser popup)
ee.Authenticate()

# Initialise with the project that holds your GEE assets
ee.Initialize(project='northstarlabs')

import pandas as pd
import numpy as np
from google.colab import files

print('✓ GEE authenticated and initialised')

✓ GEE authenticated and initialised


In [2]:
# ============================================================================
# 2. LOAD GEOGRAPHICALLY + DEPTH FILTERED PROFILES → BUILD GEE FEATURE LIST
# ============================================================================
# Input: combined_profiles_filtered.csv (output of 05_filter_profiles.R)
#   Filters already applied: lat >= 30°N, lon -170 to +60°E, depth >= 30 cm
# Climate analog filtering (MAT/MAP) is applied in Cell 5b of this notebook.

# Option A (Colab): upload the file directly
print('Upload combined_profiles_filtered.csv when prompted...')
uploaded = files.upload()
FILENAME = list(uploaded.keys())[0]

# Option B (Drive): uncomment and mount Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# FILENAME = '/content/drive/MyDrive/SoilCarbonProfiles/combined_profiles_filtered.csv'

df = pd.read_csv(FILENAME)
print(f'Loaded: {len(df):,} profiles (post geographic + depth filter)')
print(f'Datasets: {df["dataset"].value_counts().to_dict()}')
print(f'Columns: {list(df.columns)}')

# Build one ee.Feature per profile
# Only profile_id + dataset stored as GEE properties; all other attributes
# are merged back in pandas after extraction.
df_pts = df.dropna(subset=['latitude', 'longitude']).copy()
n_missing = len(df) - len(df_pts)
if n_missing > 0:
    print(f'⚠ {n_missing} rows dropped — missing latitude or longitude')

feature_list = []
for _, row in df_pts.iterrows():
    geom = ee.Geometry.Point([float(row['longitude']), float(row['latitude'])])
    feature_list.append(ee.Feature(geom, {
        'profile_id': str(row['profile_id']),
        'dataset':    str(row['dataset'])
    }))

print(f'✓ {len(feature_list):,} GEE features created (all geo+depth filtered profiles)')

Upload combined_profiles_filtered.csv when prompted...


Saving combined_profiles_filtered.csv to combined_profiles_filtered (1).csv
Loaded: 14,244 profiles (post geographic + depth filter)
Datasets: {'WOSIS 2023': 11945, 'Peat Database': 1217, 'Janousek': 1082}
Columns: ['dataset', 'profile_id', 'latitude', 'longitude', 'total_depth_cm', 'n_layers', 'sum_OrgC_Stock_kgm2', 'mean_BDOD', 'mean_OrgC_pct', 'country_name', 'year']
✓ 14,244 GEE features created (all geo+depth filtered profiles)


In [3]:
# ============================================================================
# 3. DEFINE REMOTE SENSING IMAGE STACKS
# Dates match charlies_place_v4_2.js pipeline settings
# ============================================================================

# ── DATE RANGES (match GEE JS script) ────────────────────────────
S2_START  = '2020-01-01'
S2_END    = '2023-12-31'
SAR_START = '2020-01-01'
SAR_END   = '2023-12-31'

# ── STACK 1: TOPOGRAPHY ──────────────────────────────────────────
dem          = ee.Image('NASA/NASADEM_HGT/001').select('elevation')
elevation_m  = dem.rename('elevation_m')
slope        = ee.Terrain.slope(dem).rename('slope')
elevRelMHW   = dem.subtract(0.5).rename('elevationRelMHW')

stack_topo = elevation_m.addBands(slope).addBands(elevRelMHW)
print('✓ Stack 1 — Topography: elevation_m, slope, elevationRelMHW')

# ── STACK 2: SENTINEL-1 SAR ──────────────────────────────────────
s1_col = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterDate(SAR_START, SAR_END)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
)
s1_mean  = s1_col.mean()
vv       = s1_mean.select('VV').rename('VV_mean')
vh       = s1_mean.select('VH').rename('VH_mean')
vv_vh    = s1_mean.select('VV').subtract(s1_mean.select('VH')).rename('VVVH_ratio')

stack_s1 = vv.addBands(vh).addBands(vv_vh)
print('✓ Stack 2 — Sentinel-1: VV_mean, VH_mean, VVVH_ratio')

# ── STACK 3: SENTINEL-2 OPTICAL ──────────────────────────────────
# NOTE: The S2 collection is NOT mosaicked globally here.
# Instead, extract_batch_s2() filters + mosaics per-batch inside a small
# spatial buffer around each batch of points. This is the only reliable way
# to avoid GEE computing a global mosaic on every reduceRegions() call.

S2_MAX_CLOUD   = 30    # % cloud cover threshold for initial scene filter
S2_IMAGE_LIMIT = 20    # max scenes kept per batch area (least cloudy first)
S2_BUFFER_M    = 5000  # metres buffer around each batch of points

def process_s2_image(image):
    """Cloud mask + tidal/inundation mask + scale to reflectance."""
    qa         = image.select('QA60')
    cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    ndwi_check = image.normalizedDifference(['B3', 'B8'])
    tide_mask  = ndwi_check.lt(0.1)
    return image.updateMask(cloud_mask.And(tide_mask)).divide(10000)

def make_s2_median_for_region(region_geom):
    """
    Build a spatially filtered S2 median image for a specific region.
    Called once per batch inside extract_batch_s2() so GEE only sees
    the ~20 least-cloudy scenes that intersect the batch buffer.
    """
    col = (
        ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
        .filterDate(S2_START, S2_END)
        .filterBounds(region_geom)                           # only scenes covering this area
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', S2_MAX_CLOUD))
        .limit(S2_IMAGE_LIMIT, 'CLOUDY_PIXEL_PERCENTAGE')   # keep N least-cloudy
        .map(process_s2_image)
    )
    return col.median().clip(region_geom)

print(f'✓ Stack 3 — Sentinel-2: per-batch spatial filter ({S2_IMAGE_LIMIT} images, {S2_BUFFER_M} m buffer)')
print(f'  Raw bands: B, G, R, NIR, SWIR1, SWIR2')
print(f'  Derived:   NDVI, NDBI, EVI, SAVI, LSWI, mNDWI, brightness, greenness, wetness')

# ── STACK 4: GOOGLE SATELLITE EMBEDDING V1 (64 bands, 10 m) ──────
stack_embed = (
    ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
    .median()
)
n_embed_bands = stack_embed.bandNames().size().getInfo()
print(f'✓ Stack 4 — Google Satellite Embedding V1: {n_embed_bands} bands at 10 m')

✓ Stack 1 — Topography: elevation_m, slope, elevationRelMHW
✓ Stack 2 — Sentinel-1: VV_mean, VH_mean, VVVH_ratio
✓ Stack 3 — Sentinel-2: per-batch spatial filter (20 images, 5000 m buffer)
  Raw bands: B, G, R, NIR, SWIR1, SWIR2
  Derived:   NDVI, NDBI, EVI, SAVI, LSWI, mNDWI, brightness, greenness, wetness
✓ Stack 4 — Google Satellite Embedding V1: 64 bands at 10 m


In [4]:
# ============================================================================
# 3b. DEFINE TERRACLIMATE STACK (MAT + MAP)
# Consistent with charlies_place_v4_2.js: IDAHO_EPSCOR/TERRACLIMATE, 2000-2022
# Units in raw GEE data:
#   tmmn / tmmx : °C × 10  (divide by 10 to get °C)
#   pr          : mm/month  (multiply monthly mean × 12 to get mm/year)
# ============================================================================

TC_START = '2000-01-01'
TC_END   = '2022-12-31'

terra_col = (
    ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
    .filterDate(TC_START, TC_END)
    .select(['tmmn', 'tmmx', 'pr'])
)
terra_mean = terra_col.mean()   # long-term monthly mean

# Mean Annual Temperature (°C)
# MAT = ((tmmn_mean + tmmx_mean) / 2) / 10
mat_img = terra_mean.expression(
    '((tmmn + tmmx) / 2.0) / 10.0',
    {'tmmn': terra_mean.select('tmmn'),
     'tmmx': terra_mean.select('tmmx')}
).rename('MAT_C')

# Mean Annual Precipitation (mm/year)
# mean monthly pr × 12 months
map_img = terra_mean.select('pr').multiply(12).rename('MAP_mm')

stack_climate = mat_img.addBands(map_img)
print(f'✓ Stack Climate — TerraClimate {TC_START[:4]}–{TC_END[:4]}: MAT_C (°C), MAP_mm (mm/yr)')

✓ Stack Climate — TerraClimate 2000–2022: MAT_C (°C), MAP_mm (mm/yr)


In [5]:
# ============================================================================
# 3c. SAMPLE TERRACLIMATE AT CHARLIE'S PLACE KBA CENTROID
#     → Set AOI reference MAT and MAP for climate analog filtering
# ============================================================================

# Climate analog tolerances
MAT_TOLERANCE = 2.0   # ±2°C
MAP_TOLERANCE = 0.30  # ±30%

# Load AOI from GEE asset and get its centroid
AOI_ASSET = 'projects/blue-carbon-hub/assets/Charlies_Place_KBA_boundaryFile_2025'
aoi_fc     = ee.FeatureCollection(AOI_ASSET)
aoi_geom   = aoi_fc.geometry()
aoi_centroid = aoi_geom.centroid(maxError=100)

# Sample TerraClimate at the AOI centroid
# TerraClimate native resolution ~4 km (scale=4000 m)
aoi_sample = stack_climate.sample(
    region=aoi_centroid,
    scale=4000,
    numPixels=1,
    geometries=False
).first().getInfo()

AOI_MAT = aoi_sample['properties']['MAT_C']
AOI_MAP = aoi_sample['properties']['MAP_mm']

# Compute filter bounds
MAT_MIN = AOI_MAT - MAT_TOLERANCE
MAT_MAX = AOI_MAT + MAT_TOLERANCE
MAP_MIN = AOI_MAP * (1.0 - MAP_TOLERANCE)
MAP_MAX = AOI_MAP * (1.0 + MAP_TOLERANCE)

print(f"Charlie's Place KBA — reference climate (TerraClimate {TC_START[:4]}–{TC_END[:4]})")
print(f'  MAT = {AOI_MAT:.2f} °C')
print(f'  MAP = {AOI_MAP:.0f} mm/year')
print(f'')
print(f'Climate analog filter ranges:')
print(f'  MAT: {MAT_MIN:.1f} to {MAT_MAX:.1f} °C   (±{MAT_TOLERANCE} °C)')
print(f'  MAP: {MAP_MIN:.0f} to {MAP_MAX:.0f} mm/year  (±{MAP_TOLERANCE*100:.0f}%)')

Charlie's Place KBA — reference climate (TerraClimate 2000–2022)
  MAT = 4.90 °C
  MAP = 1225 mm/year

Climate analog filter ranges:
  MAT: 2.9 to 6.9 °C   (±2.0 °C)
  MAP: 857 to 1592 mm/year  (±30%)


In [6]:
# ============================================================================
# 4. BATCH EXTRACTION FUNCTIONS
# ============================================================================
# Two functions:
#
#   extract_batch()     — general purpose: used for Topo, SAR, TerraClimate,
#                         and Embedding. Accepts a pre-built ee.Image.
#                         clip_buffer_m=0 means no clipping (global layers).
#
#   extract_batch_s2()  — Sentinel-2 specific: rebuilds a spatially filtered
#                         S2 median from scratch for each batch's bounding box.
#                         This is the only reliable way to avoid GEE computing
#                         a full global mosaic per call, which causes timeouts
#                         even with batch_size=8 and tileScale=4.
#
# Both use reduceRegions() in WGS84, avoiding the UTM Zone projection error
# that broke the GEE JS sampleRegions() call for global Janousek points.
# ============================================================================

def extract_batch(image, features, name, batch_size=100, scale=30, clip_buffer_m=0):
    """Extract pixel values at point locations using batched reduceRegions."""
    results   = []
    n_batches = (len(features) + batch_size - 1) // batch_size
    n_failed  = 0
    print(f'--- Extracting {name} ({len(features):,} pts, batch={batch_size}, scale={scale}m) ---')

    for i in range(0, len(features), batch_size):
        batch     = features[i: i + batch_size]
        fc        = ee.FeatureCollection(batch)
        batch_num = i // batch_size + 1
        try:
            img = image.clip(fc.geometry().buffer(clip_buffer_m)) if clip_buffer_m > 0 else image
            res = img.reduceRegions(
                collection = fc,
                reducer    = ee.Reducer.first(),
                scale      = scale,
                tileScale  = 2
            )
            data = res.getInfo()['features']
            for f in data:
                results.append(f['properties'])
            if batch_num % 10 == 0 or batch_num == n_batches:
                print(f'   Batch {batch_num}/{n_batches} OK  ({len(results):,} rows so far)')
        except Exception as e:
            n_failed += 1
            print(f'   Batch {batch_num}/{n_batches} FAILED: {e}')

    print(f'✓ {name} complete — {len(results):,} rows, {n_failed} batches failed')
    return pd.DataFrame(results)


def extract_batch_s2(features, name, band_selector_fn,
                     batch_size=25, scale=30, buffer_m=None):
    """
    Sentinel-2 specific extraction.

    For each batch:
      1. Compute a bounding box + buffer around the batch points (server-side)
      2. Call make_s2_median_for_region() to build a spatially filtered median
         using only the ~20 least-cloudy S2 scenes covering that area
      3. Apply band_selector_fn to pick raw or derived bands
      4. reduceRegions() on just those points

    This ensures GEE never touches more S2 imagery than needed for
    the small geographic footprint of each batch.

    band_selector_fn : callable(s2_median_image) → ee.Image with renamed bands
    buffer_m         : spatial buffer around batch bbox (defaults to S2_BUFFER_M)
    """
    if buffer_m is None:
        buffer_m = S2_BUFFER_M

    results   = []
    n_batches = (len(features) + batch_size - 1) // batch_size
    n_failed  = 0
    print(f'--- Extracting {name} ({len(features):,} pts, batch={batch_size}, '
          f'scale={scale}m, buffer={buffer_m}m) ---')

    for i in range(0, len(features), batch_size):
        batch     = features[i: i + batch_size]
        fc        = ee.FeatureCollection(batch)
        batch_num = i // batch_size + 1
        try:
            # Build a buffered region around this batch
            region = fc.geometry().bounds().buffer(buffer_m)

            # Build a spatially filtered S2 median for this region only
            s2_local = make_s2_median_for_region(region)

            # Select / compute the desired bands
            img = band_selector_fn(s2_local)

            res = img.reduceRegions(
                collection = fc,
                reducer    = ee.Reducer.first(),
                scale      = scale,
                tileScale  = 2
            )
            data = res.getInfo()['features']
            for f in data:
                results.append(f['properties'])
            if batch_num % 10 == 0 or batch_num == n_batches:
                print(f'   Batch {batch_num}/{n_batches} OK  ({len(results):,} rows so far)')
        except Exception as e:
            n_failed += 1
            print(f'   Batch {batch_num}/{n_batches} FAILED: {e}')

    print(f'✓ {name} complete — {len(results):,} rows, {n_failed} batches failed')
    return pd.DataFrame(results)


# ── Band selector functions for S2 ───────────────────────────────
def s2_raw_bands(s2):
    """Select 6 raw reflectance bands from a per-batch S2 median image."""
    return (
        s2.select('B2').rename('B')
        .addBands(s2.select('B3').rename('G'))
        .addBands(s2.select('B4').rename('R'))
        .addBands(s2.select('B8').rename('NIR'))
        .addBands(s2.select('B11').rename('SWIR1'))
        .addBands(s2.select('B12').rename('SWIR2'))
    )

def s2_derived_bands(s2):
    """Compute 9 derived indices + Tasseled Cap from a per-batch S2 median image."""
    B     = s2.select('B2')
    G     = s2.select('B3')
    R     = s2.select('B4')
    NIR   = s2.select('B8')
    SWIR1 = s2.select('B11')
    SWIR2 = s2.select('B12')

    ndvi  = s2.normalizedDifference(['B8', 'B4']).rename('NDVI_median')
    ndbi  = s2.normalizedDifference(['B11', 'B8']).rename('NDBI_median')
    evi   = s2.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': NIR, 'RED': R, 'BLUE': B}
    ).rename('EVI_median')
    savi  = s2.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
        {'NIR': NIR, 'RED': R}
    ).rename('SAVI_median')
    lswi  = s2.normalizedDifference(['B8', 'B11']).rename('LSWI_median')
    mndwi = s2.normalizedDifference(['B3', 'B11']).rename('mNDWI_median')

    brightness = s2.expression(
        '0.3037*B2 + 0.2793*B3 + 0.4743*B4 + 0.5585*B8 + 0.5082*B11 + 0.1863*B12',
        {'B2': B, 'B3': G, 'B4': R, 'B8': NIR, 'B11': SWIR1, 'B12': SWIR2}
    ).rename('brightness')
    greenness = s2.expression(
        '-0.2848*B2 - 0.2435*B3 - 0.5436*B4 + 0.7243*B8 + 0.0840*B11 - 0.1800*B12',
        {'B2': B, 'B3': G, 'B4': R, 'B8': NIR, 'B11': SWIR1, 'B12': SWIR2}
    ).rename('greenness')
    wetness = s2.expression(
        '0.1509*B2 + 0.1973*B3 + 0.3279*B4 + 0.3406*B8 - 0.7112*B11 - 0.4572*B12',
        {'B2': B, 'B3': G, 'B4': R, 'B8': NIR, 'B11': SWIR1, 'B12': SWIR2}
    ).rename('wetness')

    return (
        ndvi.addBands(ndbi).addBands(evi).addBands(savi)
        .addBands(lswi).addBands(mndwi)
        .addBands(brightness).addBands(greenness).addBands(wetness)
    )

print('✓ extract_batch() and extract_batch_s2() defined')

✓ extract_batch() and extract_batch_s2() defined


In [7]:
# ============================================================================
# 5. EXTRACT TERRACLIMATE (MAT + MAP) AT ALL GEO+DEPTH FILTERED POINTS
# Extracted FIRST (before Topo/S1/S2) at ~4 km resolution — cheap and fast.
# Climate filter is applied immediately after (Cell 5b) to reduce the point
# set before the expensive Sentinel-2 extraction runs (Cell 5d).
# ============================================================================
df_climate = extract_batch(stack_climate, feature_list, 'TerraClimate (MAT/MAP)',
                           batch_size=500, scale=4000, clip_buffer_m=0)

print(f'\nTerraClimate extraction complete: {len(df_climate):,} rows')
if 'MAT_C' in df_climate.columns:
    print('\nMAT_C (°C) distribution:')
    print(df_climate['MAT_C'].describe().round(2))
    print('\nMAP_mm distribution:')
    print(df_climate['MAP_mm'].describe().round(0))

--- Extracting TerraClimate (MAT/MAP) (14,244 pts, batch=500, scale=4000m) ---
   Batch 10/29 OK  (5,000 rows so far)
   Batch 20/29 OK  (10,000 rows so far)
   Batch 29/29 OK  (14,244 rows so far)
✓ TerraClimate (MAT/MAP) complete — 14,244 rows, 0 batches failed

TerraClimate extraction complete: 14,244 rows

MAT_C (°C) distribution:
count    14197.00
mean         9.97
std          6.05
min        -17.41
25%          6.69
50%         10.61
75%         13.97
max         24.24
Name: MAT_C, dtype: float64

MAP_mm distribution:
count    14197.0
mean       840.0
std        531.0
min         55.0
25%        399.0
50%        778.0
75%       1178.0
max       3547.0
Name: MAP_mm, dtype: float64


In [8]:
# ============================================================================
# 5b. APPLY CLIMATE ANALOG FILTER
#     Keep profiles whose MAT and MAP are within tolerance of AOI
# ============================================================================
def merge_df(main, sub, key='profile_id'):
    """Left-join extracted values onto the main profiles dataframe."""
    if sub is None or sub.empty:
        print('  ⚠ Skipping empty extraction result')
        return main
    drop_cols = ['system:index', 'dataset']
    keep_cols = [key] + [c for c in sub.columns if c not in [key] + drop_cols]
    return pd.merge(main, sub[keep_cols], on=key, how='left')


# Merge climate values onto profile dataframe
df_climate['profile_id'] = df_climate['profile_id'].astype(str)
df_with_climate = df_pts.copy()
df_with_climate['profile_id'] = df_with_climate['profile_id'].astype(str)
df_with_climate = merge_df(df_with_climate, df_climate)

n_has_climate = df_with_climate['MAT_C'].notna().sum() if 'MAT_C' in df_with_climate.columns else 0
print(f'Profiles with TerraClimate data: {n_has_climate:,} / {len(df_with_climate):,}')

# Climate filter:
#   - Keep if MAT AND MAP are within bounds
#   - Also keep if climate data is missing (NaN) — don't exclude unknowns
if 'MAT_C' in df_with_climate.columns:
    within_mat = (df_with_climate['MAT_C'] >= MAT_MIN) & (df_with_climate['MAT_C'] <= MAT_MAX)
    within_map = (df_with_climate['MAP_mm'] >= MAP_MIN) & (df_with_climate['MAP_mm'] <= MAP_MAX)
    climate_pass = within_mat & within_map
    no_data      = df_with_climate['MAT_C'].isna()
    keep_mask    = climate_pass | no_data
else:
    print('⚠ MAT_C column not found — skipping climate filter (check TerraClimate extraction)')
    keep_mask = pd.Series([True] * len(df_with_climate), index=df_with_climate.index)

df_filtered = df_with_climate[keep_mask].copy()
df_removed  = df_with_climate[~keep_mask].copy()

print(f'\n── Climate filter summary ──────────────────────────────────')
print(f'  Input (post geo+depth filter):  {len(df_with_climate):,} profiles')
print(f'  AOI MAT = {AOI_MAT:.2f} °C  →  keep {MAT_MIN:.1f}–{MAT_MAX:.1f} °C')
print(f'  AOI MAP = {AOI_MAP:.0f} mm  →  keep {MAP_MIN:.0f}–{MAP_MAX:.0f} mm/yr')
print(f'  Removed (outside climate range): {len(df_removed):,}')
print(f'  Retained:                        {len(df_filtered):,}')
print(f'\nRemoved by dataset:')
print(df_removed['dataset'].value_counts().to_string() if len(df_removed) > 0 else '  (none)')

# Save audit file of removed profiles
AUDIT_FILE = 'CorePoints_Climate_Filtered_OUT.csv'
df_removed.to_csv(AUDIT_FILE, index=False)
files.download(AUDIT_FILE)
print(f'\n✓ Audit file saved: {AUDIT_FILE}  ({len(df_removed):,} removed profiles)')

Profiles with TerraClimate data: 15,923 / 15,996

── Climate filter summary ──────────────────────────────────
  Input (post geo+depth filter):  15,996 profiles
  AOI MAT = 4.90 °C  →  keep 2.9–6.9 °C
  AOI MAP = 1225 mm  →  keep 857–1592 mm/yr
  Removed (outside climate range): 15,082
  Retained:                        914

Removed by dataset:
dataset
WOSIS 2023       11533
Peat Database     1807
Janousek          1742


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Audit file saved: CorePoints_Climate_Filtered_OUT.csv  (15,082 removed profiles)


In [9]:
# ============================================================================
# 5c. REBUILD FEATURE LIST FROM CLIMATE-FILTERED PROFILES
#     All subsequent extractions use this filtered list
# ============================================================================
feature_list_filtered = []
for _, row in df_filtered.iterrows():
    geom = ee.Geometry.Point([float(row['longitude']), float(row['latitude'])])
    feature_list_filtered.append(ee.Feature(geom, {
        'profile_id': str(row['profile_id']),
        'dataset':    str(row['dataset'])
    }))

print(f'✓ Climate-filtered feature list: {len(feature_list_filtered):,} points')
print(f'  (Reduced from {len(feature_list):,} geo+depth filtered points)')
print(f'  Breakdown: {df_filtered["dataset"].value_counts().to_dict()}')

✓ Climate-filtered feature list: 914 points
  (Reduced from 14,244 geo+depth filtered points)
  Breakdown: {'WOSIS 2023': 412, 'Peat Database': 286, 'Janousek': 216}


In [11]:
# ============================================================================
# 5d. EXTRACT REMOTE SENSING STACKS AT CLIMATE-FILTERED POINTS
# ============================================================================
# Topo and SAR use extract_batch() — simple global layers, no S2 complexity.
# Sentinel-2 uses extract_batch_s2() — builds a fresh spatially filtered
# median per batch so GEE never touches a global S2 mosaic.
# ============================================================================

# Stack 1: Topography — static global DEM, no clipping needed
df_topo = extract_batch(stack_topo, feature_list_filtered, 'Topography',
                        batch_size=500, scale=30, clip_buffer_m=0)

# Stack 2: Sentinel-1 SAR — global mean, no clip needed
df_s1   = extract_batch(stack_s1, feature_list_filtered, 'Sentinel-1',
                        batch_size=100, scale=30, clip_buffer_m=0)

# Stack 3a: Sentinel-2 raw reflectance (6 bands)
# extract_batch_s2() filters the collection to the batch region internally,
# so only ~20 local scenes are ever processed per batch.
df_s2_raw = extract_batch_s2(
    feature_list_filtered, 'Sentinel-2 Raw', s2_raw_bands,
    batch_size=1, scale=30
)

# Stack 3b: Sentinel-2 derived indices + Tasseled Cap (9 bands)
# Separate call — same per-batch approach, independent S2 median built each time
df_s2_derived = extract_batch_s2(
    feature_list_filtered, 'Sentinel-2 Derived', s2_derived_bands,
    batch_size=1, scale=30
)

print('\n=== Remote sensing extraction complete ===')
print(f'  Topo:               {len(df_topo):,} rows | {len(df_topo.columns)} cols')
print(f'  Sentinel-1:         {len(df_s1):,} rows | {len(df_s1.columns)} cols')
print(f'  Sentinel-2 Raw:     {len(df_s2_raw):,} rows | {len(df_s2_raw.columns)} cols')
print(f'  Sentinel-2 Derived: {len(df_s2_derived):,} rows | {len(df_s2_derived.columns)} cols')

--- Extracting Topography (914 pts, batch=500, scale=30m) ---
   Batch 2/2 OK  (914 rows so far)
✓ Topography complete — 914 rows, 0 batches failed
--- Extracting Sentinel-1 (914 pts, batch=100, scale=30m) ---
   Batch 10/10 OK  (914 rows so far)
✓ Sentinel-1 complete — 914 rows, 0 batches failed
--- Extracting Sentinel-2 Raw (914 pts, batch=1, scale=30m, buffer=5000m) ---
   Batch 10/914 OK  (10 rows so far)
   Batch 20/914 OK  (20 rows so far)
   Batch 30/914 OK  (30 rows so far)
   Batch 40/914 OK  (40 rows so far)
   Batch 50/914 OK  (50 rows so far)
   Batch 60/914 OK  (60 rows so far)
   Batch 70/914 OK  (70 rows so far)
   Batch 80/914 OK  (80 rows so far)
   Batch 90/914 OK  (90 rows so far)
   Batch 100/914 OK  (100 rows so far)
   Batch 110/914 OK  (110 rows so far)
   Batch 120/914 OK  (120 rows so far)
   Batch 130/914 OK  (130 rows so far)
   Batch 140/914 OK  (140 rows so far)
   Batch 150/914 OK  (150 rows so far)
   Batch 160/914 OK  (160 rows so far)
   Batch 170/914 O

   Batch 530/914 OK  (530 rows so far)
   Batch 540/914 OK  (540 rows so far)
   Batch 550/914 OK  (550 rows so far)
   Batch 560/914 OK  (560 rows so far)
   Batch 570/914 OK  (570 rows so far)
   Batch 580/914 OK  (580 rows so far)
   Batch 590/914 OK  (590 rows so far)
   Batch 600/914 OK  (600 rows so far)
   Batch 610/914 OK  (610 rows so far)
   Batch 620/914 OK  (620 rows so far)
   Batch 630/914 OK  (630 rows so far)
   Batch 640/914 OK  (640 rows so far)
   Batch 650/914 OK  (650 rows so far)
   Batch 660/914 OK  (660 rows so far)
   Batch 670/914 OK  (670 rows so far)
   Batch 680/914 OK  (680 rows so far)
   Batch 690/914 OK  (690 rows so far)
   Batch 700/914 OK  (700 rows so far)
   Batch 710/914 OK  (710 rows so far)
   Batch 720/914 OK  (720 rows so far)
   Batch 730/914 OK  (730 rows so far)
   Batch 740/914 OK  (740 rows so far)
   Batch 750/914 OK  (750 rows so far)
   Batch 760/914 OK  (760 rows so far)
   Batch 770/914 OK  (770 rows so far)
   Batch 780/914 OK  (780

In [15]:
# ============================================================================
# 6. CLUSTER-THEN-MATCH — AOI K-MEANS + EMBEDDING SIMILARITY
# ============================================================================
# Strategy
# --------
# Instead of exporting all 64 raw embedding bands, we:
#   A. Sample all AOI pixels at 30 m → run K-means (GEE server-side, k=5)
#   B. Extract the medoid vector for each cluster (pixel closest to mean)
#      → downloads only k × 64 values from GEE
#   C. Extract 64-band embedding at each climate-filtered core location
#   D. Compute cosine similarity (Python/numpy) — cores vs. medoids
#      → N_cores × k similarity scores used as transfer learning weights
#   E. Build cluster-label + similarity raster image for GeoTIFF export
#
# Outputs (built here, exported in Cell 7):
#   sim_df          — profile_id + embedding_sim_to_cluster_0..4 +
#                     embedding_max_sim + embedding_best_cluster
#   aoi_output_img  — 2-band GEE image (cluster_id, sim_to_nearest_medoid)
#   medoids_data    — raw getInfo() result (k rows × 64 bands)
# ============================================================================

import numpy as np

# ── USER-TUNABLE CONFIGURATION ────────────────────────────────────
N_CLUSTERS     = 5    # k for K-means on AOI pixels
AOI_GRID_SCALE = 30   # metres — pixel spacing for AOI pixel sampling
EMBED_SCALE    = 10   # metres — native resolution of Google Embedding V1

print(f'Cluster-then-Match config: k={N_CLUSTERS}, AOI grid={AOI_GRID_SCALE}m, embed={EMBED_SCALE}m')

# ═══════════════════════════════════════════════════════════════════
# STEP A: Sample AOI pixels + run K-means (GEE server-side)
# ═══════════════════════════════════════════════════════════════════
print('\n── Step A: Sampling AOI pixels at 30 m and running K-means ──')

# Sample all pixels inside the AOI on a regular 30 m grid.
# geometries=True keeps coordinates so we can export the cluster image.
aoi_pixels = stack_embed.sample(
    region     = aoi_geom,
    scale      = AOI_GRID_SCALE,
    seed       = 42,
    geometries = True
)

n_aoi_pixels = aoi_pixels.size().getInfo()
print(f'  AOI pixels sampled: {n_aoi_pixels:,}')

# Train K-means clusterer on the sampled AOI pixels (server-side)
clusterer   = ee.Clusterer.wekaKMeans(N_CLUSTERS, seed=42).train(aoi_pixels)

# Apply cluster labels back to the sampled pixels (FeatureCollection)
aoi_clustered = aoi_pixels.cluster(clusterer)   # adds 'cluster' property to each Feature

# Apply cluster labels as a raster for GeoTIFF export
aoi_cluster_img = stack_embed.cluster(clusterer).rename('cluster_id')

print(f'  K-means complete: {N_CLUSTERS} clusters assigned to AOI pixels')

# ═══════════════════════════════════════════════════════════════════
# STEP B: Extract mean embedding vector per cluster
#   Mean vector computed server-side via reduceColumns.
#   Downloads only k × 64 values from GEE.
# ═══════════════════════════════════════════════════════════════════
print(f'\n── Step B: Extracting mean embedding vector for each cluster ──')

embed_band_names = stack_embed.bandNames().getInfo()

cluster_means = {}
for cluster_id in range(N_CLUSTERS):
    cluster_pts = aoi_clustered.filter(ee.Filter.eq('cluster', cluster_id))

    mean_dict = cluster_pts.reduceColumns(
        reducer   = ee.Reducer.mean().repeat(len(embed_band_names)),
        selectors = embed_band_names
    ).getInfo()

    cluster_means[cluster_id] = mean_dict['mean']
    print(f'  Cluster {cluster_id}: {len(mean_dict["mean"])} embedding bands extracted')

print(f'  Done — {len(cluster_means)} cluster mean vectors ready')

# ═══════════════════════════════════════════════════════════════════
# STEP C: Extract 64-band embedding at each climate-filtered core
# ═══════════════════════════════════════════════════════════════════
print(f'\n── Step C: Extracting 64-band embedding at {len(feature_list_filtered):,} core locations ──')

df_embed = extract_batch(stack_embed, feature_list_filtered, 'Google Embedding V1',
                         batch_size=50, scale=EMBED_SCALE, clip_buffer_m=0)

print(f'  Core embeddings: {len(df_embed):,} rows | {len(df_embed.columns)} cols')
embed_cols = [c for c in df_embed.columns
              if c not in ['profile_id', 'dataset', 'system:index']]
print(f'  Embedding bands ({len(embed_cols)}): {embed_cols[:4]} … {embed_cols[-2:]}')

# ═══════════════════════════════════════════════════════════════════
# STEP D: Compute cosine similarity (Python/numpy)
#   cosine_sim(core_i, mean_j) = dot(norm(core_i), norm(mean_j))
# ═══════════════════════════════════════════════════════════════════
print(f'\n── Step D: Computing cosine similarity ({len(df_embed):,} cores × {N_CLUSTERS} cluster means) ──')

# Core embedding matrix  — N_cores × 64
core_matrix = df_embed[embed_cols].values.astype(float)

# Cluster mean embedding matrix — N_CLUSTERS × 64
mean_matrix = np.array([
    cluster_means[j]
    for j in range(N_CLUSTERS)
], dtype=float)

def l2_norm(mat):
    """Row-wise L2 normalisation; zero-vectors left at zero (avoid /0)."""
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms

core_norm = l2_norm(np.nan_to_num(core_matrix, nan=0.0))
mean_norm = l2_norm(np.nan_to_num(mean_matrix, nan=0.0))

# Cosine similarity: N_cores × N_CLUSTERS
sim_matrix = core_norm @ mean_norm.T

# Build output dataframe
sim_df = df_embed[['profile_id']].copy()
for j in range(N_CLUSTERS):
    sim_df[f'embedding_sim_to_cluster_{j}'] = sim_matrix[:, j]
sim_df['embedding_max_sim']      = sim_matrix.max(axis=1)
sim_df['embedding_best_cluster'] = sim_matrix.argmax(axis=1)

print(f'  sim_df shape: {sim_df.shape}')
print(f'  embedding_max_sim summary:')
print(sim_df['embedding_max_sim'].describe().round(4).to_string())
print(f'\n  Best-cluster distribution:')
print(sim_df['embedding_best_cluster'].value_counts().sort_index().to_string())
# ═══════════════════════════════════════════════════════════════════
# STEP E: Build per-pixel cluster-similarity raster (for GeoTIFF)
#   Band 1: cluster_id (0–k-1)
#   Band 2: cosine similarity of each pixel to its cluster's mean vector
# ═══════════════════════════════════════════════════════════════════
print(f'\n── Step E: Building cluster + similarity raster over AOI ──')

def cosine_sim_image(mean_vec, band_names):
    """
    Return a single-band GEE image of cosine similarity between
    stack_embed and one cluster mean vector, over the full AOI extent.
    """
    mean_img = ee.Image.constant(mean_vec).rename(band_names)

    # dot product (numerator)
    dot = stack_embed.multiply(mean_img).reduce(ee.Reducer.sum())

    # L2 norm of embedding at each pixel (denominator term 1)
    norm_emb = stack_embed.pow(2).reduce(ee.Reducer.sum()).sqrt()

    # L2 norm of mean vector (denominator term 2 — scalar constant)
    norm_mean = ee.Number(sum(v ** 2 for v in mean_vec)) \
                         .sqrt().max(ee.Number(1e-10))

    return dot.divide(norm_emb.max(ee.Number(1e-10))).divide(norm_mean)

# Mosaic: for each pixel, show similarity to its own cluster's mean vector
sim_layers = [
    cosine_sim_image(cluster_means[i], embed_cols)
    .updateMask(aoi_cluster_img.eq(i))
    .rename('sim_to_cluster_mean')
    for i in range(N_CLUSTERS)
]
sim_to_nearest = ee.ImageCollection(sim_layers).mosaic()

# 2-band output image: cluster_id (Int16) + similarity (Float32)
aoi_output_img = (
    aoi_cluster_img.toInt16()
    .addBands(sim_to_nearest.toFloat())
    .clip(aoi_geom)
)

print(f'  aoi_output_img bands: cluster_id (Int16), sim_to_cluster_mean (Float32)')
print(f'  Ready for GeoTIFF export in Cell 7.')
print(f'\n✓ Cluster-then-Match complete.')

Cluster-then-Match config: k=5, AOI grid=30m, embed=10m

── Step A: Sampling AOI pixels at 30 m and running K-means ──
  AOI pixels sampled: 142,738
  K-means complete: 5 clusters assigned to AOI pixels

── Step B: Extracting mean embedding vector for each cluster ──
  Cluster 0: 64 embedding bands extracted
  Cluster 1: 64 embedding bands extracted
  Cluster 2: 64 embedding bands extracted
  Cluster 3: 64 embedding bands extracted
  Cluster 4: 64 embedding bands extracted
  Done — 5 cluster mean vectors ready

── Step C: Extracting 64-band embedding at 914 core locations ──
--- Extracting Google Embedding V1 (914 pts, batch=50, scale=10m) ---
   Batch 10/19 OK  (500 rows so far)
   Batch 19/19 OK  (914 rows so far)
✓ Google Embedding V1 complete — 914 rows, 0 batches failed
  Core embeddings: 914 rows | 66 cols
  Embedding bands (64): ['A00', 'A01', 'A02', 'A03'] … ['A62', 'A63']

── Step D: Computing cosine similarity (914 cores × 5 cluster means) ──
  sim_df shape: (914, 8)
  embedd

In [17]:
# ============================================================================
# 7. MERGE ALL RESULTS + EXPORT OUTPUT FILES
# ============================================================================
# Outputs:
#   CorePoints_Covariates_CSV.csv          — Topo + SAR + S2 + TerraClimate
#   CorePoints_EmbeddingSimilarity_CSV.csv — cosine sim to each AOI cluster
#   AOI_ClusterMeans.csv                   — k × 64 cluster mean embedding vectors
#   AOI_KMeans_Clusters.tif                — 2-band raster (cluster_id + sim)
#                                            exported to Google Drive
# ============================================================================

# ── File 1: Covariates CSV (Topo + SAR + S2 + TerraClimate) ─────
final_cov = df_filtered.copy()
final_cov['profile_id'] = final_cov['profile_id'].astype(str)

final_cov = merge_df(final_cov, df_topo)
final_cov = merge_df(final_cov, df_s1)
final_cov = merge_df(final_cov, df_s2_raw)
final_cov = merge_df(final_cov, df_s2_derived)

print(f'CorePoints_Covariates_CSV: {len(final_cov):,} rows × {len(final_cov.columns)} cols')

check_cols = [c for c in ['elevation_m', 'NDVI_median', 'VV_mean', 'MAT_C', 'B']
              if c in final_cov.columns]
if check_cols:
    n_covered = final_cov[check_cols].notna().any(axis=1).sum()
    print(f'  Profiles with ≥1 covariate: {n_covered:,} / {len(final_cov):,} '
          f'({100*n_covered/len(final_cov):.1f}%)')
    print('\nNA rates by dataset (key covariates):')
    for col in check_cols:
        na_rates = final_cov.groupby('dataset')[col].apply(
            lambda x: x.isna().mean()).round(3)
        print(f'  {col}: {na_rates.to_dict()}')

COV_OUTFILE = 'CorePoints_Covariates_CSV.csv'
final_cov.to_csv(COV_OUTFILE, index=False)
files.download(COV_OUTFILE)
print(f'\n✓ Saved: {COV_OUTFILE}')


# ── File 2: Embedding Similarity CSV ─────────────────────────────
sim_df['profile_id'] = sim_df['profile_id'].astype(str)

sim_df_out = pd.merge(
    df_filtered[['profile_id', 'dataset', 'latitude', 'longitude']].assign(
        profile_id=lambda d: d['profile_id'].astype(str)
    ),
    sim_df,
    on='profile_id',
    how='left'
)

print(f'\nCorePoints_EmbeddingSimilarity_CSV: '
      f'{len(sim_df_out):,} rows × {len(sim_df_out.columns)} cols')
sim_cols = [c for c in sim_df_out.columns if c.startswith('embedding_')]
print(f'  Similarity columns ({len(sim_cols)}): {sim_cols}')
n_with_sim = sim_df_out['embedding_max_sim'].notna().sum()
print(f'  Cores with similarity values: {n_with_sim:,} / {len(sim_df_out):,} '
      f'({100*n_with_sim/len(sim_df_out):.1f}%)')

SIM_OUTFILE = 'CorePoints_EmbeddingSimilarity_CSV.csv'
sim_df_out.to_csv(SIM_OUTFILE, index=False)
files.download(SIM_OUTFILE)
print(f'✓ Saved: {SIM_OUTFILE}')


# ── File 3: AOI Cluster Mean Vectors CSV ─────────────────────────
# k rows × 64 embedding bands — the mean vectors per AOI cluster.
# Useful for inspecting what each cluster looks like spectrally.
mean_rows = []
for cluster_id, mean_vec in cluster_means.items():
    row = {'cluster_id': cluster_id}
    row.update({b: v for b, v in zip(embed_cols, mean_vec)})
    mean_rows.append(row)

df_cluster_means = pd.DataFrame(mean_rows).sort_values('cluster_id').reset_index(drop=True)
print(f'\nAOI_ClusterMeans: {len(df_cluster_means)} rows × {len(df_cluster_means.columns)} cols '
      f'(1 row per cluster, {len(embed_cols)} embedding bands)')

MEANS_OUTFILE = 'AOI_ClusterMeans.csv'
df_cluster_means.to_csv(MEANS_OUTFILE, index=False)
files.download(MEANS_OUTFILE)
print(f'✓ Saved: {MEANS_OUTFILE}')


# ── File 4: AOI K-Means Cluster GeoTIFF → Google Drive ───────────
# 2-band image:
#   Band 1 (cluster_id)        : integer cluster label 0 – (k-1)
#   Band 2 (sim_to_cluster_mean): cosine similarity to own-cluster mean vector
# CRS: EPSG:4326, scale: 30 m
GEOTIFF_NAME = 'AOI_KMeans_Clusters'

export_task = ee.batch.Export.image.toDrive(
    image          = aoi_output_img,
    description    = GEOTIFF_NAME,
    folder         = 'GEE_Exports',
    fileNamePrefix = GEOTIFF_NAME,
    region         = aoi_geom,
    scale          = AOI_GRID_SCALE,
    crs            = 'EPSG:4326',
    maxPixels      = 1e9,
    fileFormat     = 'GeoTIFF'
)
export_task.start()
print(f'\n✓ GeoTIFF export task started: "{GEOTIFF_NAME}"')
print(f'  Bands: cluster_id (Int16), sim_to_cluster_mean (Float32)')
print(f'  Scale: {AOI_GRID_SCALE} m | CRS: EPSG:4326')
print(f'  Destination: Google Drive → GEE_Exports/{GEOTIFF_NAME}.tif')
print(f'  Monitor at: https://code.earthengine.google.com/  (Tasks tab)')
print(f'  Status: export_task.status()')


# ── Summary ───────────────────────────────────────────────────────
print('\n' + '═' * 62)
print('=== Extraction + Cluster-then-Match complete ===')
print(f'  {COV_OUTFILE}')
print(f'    → {len(final_cov):,} profiles × {len(final_cov.columns)} cols')
print(f'  {SIM_OUTFILE}')
print(f'    → {len(sim_df_out):,} profiles × {len(sim_df_out.columns)} cols')
print(f'       (use embedding_max_sim as transfer-learning weight in R)')
print(f'  {MEANS_OUTFILE}')
print(f'    → {len(df_cluster_means)} cluster mean vectors × {len(embed_cols)} embedding bands')
print(f'  {GEOTIFF_NAME}.tif')
print(f'    → GDrive/GEE_Exports/  (2-band: cluster_id, sim_to_cluster_mean)')
print(f'  CorePoints_Climate_Filtered_OUT.csv')
print(f'    → {len(df_removed):,} profiles removed by climate filter')
print('═' * 62)

CorePoints_Covariates_CSV: 11,684 rows × 34 cols
  Profiles with ≥1 covariate: 11,682 / 11,684 (100.0%)

NA rates by dataset (key covariates):
  elevation_m: {'Janousek': 0.001, 'Peat Database': 0.001, 'WOSIS 2023': 0.007}
  NDVI_median: {'Janousek': 0.051, 'Peat Database': 0.051, 'WOSIS 2023': 0.0}
  VV_mean: {'Janousek': 0.001, 'Peat Database': 0.001, 'WOSIS 2023': 0.0}
  MAT_C: {'Janousek': 0.292, 'Peat Database': 0.285, 'WOSIS 2023': 0.01}
  B: {'Janousek': 0.051, 'Peat Database': 0.051, 'WOSIS 2023': 0.0}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Saved: CorePoints_Covariates_CSV.csv

CorePoints_EmbeddingSimilarity_CSV: 1,352 rows × 11 cols
  Similarity columns (7): ['embedding_sim_to_cluster_0', 'embedding_sim_to_cluster_1', 'embedding_sim_to_cluster_2', 'embedding_sim_to_cluster_3', 'embedding_sim_to_cluster_4', 'embedding_max_sim', 'embedding_best_cluster']
  Cores with similarity values: 1,352 / 1,352 (100.0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Saved: CorePoints_EmbeddingSimilarity_CSV.csv

AOI_ClusterMeans: 5 rows × 65 cols (1 row per cluster, 64 embedding bands)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Saved: AOI_ClusterMeans.csv

✓ GeoTIFF export task started: "AOI_KMeans_Clusters"
  Bands: cluster_id (Int16), sim_to_cluster_mean (Float32)
  Scale: 30 m | CRS: EPSG:4326
  Destination: Google Drive → GEE_Exports/AOI_KMeans_Clusters.tif
  Monitor at: https://code.earthengine.google.com/  (Tasks tab)
  Status: export_task.status()

══════════════════════════════════════════════════════════════
=== Extraction + Cluster-then-Match complete ===
  CorePoints_Covariates_CSV.csv
    → 11,684 profiles × 34 cols
  CorePoints_EmbeddingSimilarity_CSV.csv
    → 1,352 profiles × 11 cols
       (use embedding_max_sim as transfer-learning weight in R)
  AOI_ClusterMeans.csv
    → 5 cluster mean vectors × 64 embedding bands
  AOI_KMeans_Clusters.tif
    → GDrive/GEE_Exports/  (2-band: cluster_id, sim_to_cluster_mean)
  CorePoints_Climate_Filtered_OUT.csv
    → 15,082 profiles removed by climate filter
══════════════════════════════════════════════════════════════


---
## Troubleshooting

### Batch timeouts / `EEException: Computation timed out`
The S2 extractor (`extract_batch_s2`) builds a fresh, spatially filtered median for each batch so GEE only processes ~20 local scenes. If timeouts still occur:
- Reduce `batch_size` from 25 → 10 in the `extract_batch_s2()` calls in Cell 5d
- Reduce `S2_IMAGE_LIMIT` from 20 → 10 in Cell 3
- Reduce `S2_BUFFER_M` from 5000 → 2000 in Cell 3
- For the Embedding extractor, try `batch_size=10` if 50 fails

### Why `extract_batch_s2()` instead of a pre-built mosaic
The previous approach pre-built a global S2 mosaic with `.limit(50)` and then called `reduceRegions()`. GEE ignores the image limit when computing a global mosaic — it still assembles the full collection graph before evaluating any pixels. The result was a timeout even at `batch_size=8`, because every call triggered computation over thousands of scenes globally.

`extract_batch_s2()` fixes this by calling `.filterBounds(region)` *before* the median, so GEE's lazy evaluation only ever materialises the ~20 scenes that actually overlap the batch bounding box.

### `EEException: User memory limit exceeded`
Remaining optimisations in place:
1. **Cell reorder** — TerraClimate extracted first (Cell 5), climate filter applied (Cell 5b), so S2 runs on ~40–53% fewer points (Cell 5d)
2. **S2 band split** — raw (6 bands) and derived (9 bands) extracted in separate calls
3. **Per-batch spatial filter** — S2 collection filtered to batch region before median
4. **tileScale=2** — splits computation tiles to reduce per-tile memory

### Climate filter removed too many / too few profiles
- Adjust `MAT_TOLERANCE` and `MAP_TOLERANCE` in Cell 3c
- Check `CorePoints_Climate_Filtered_OUT.csv` to see which profiles were removed
- Re-run from Cell 5b onwards (no need to re-extract TerraClimate)

### AOI centroid climate sample returns None
- Verify `AOI_ASSET` path in Cell 3c
- Fallback: hardcode `AOI_MAT = 4.5` and `AOI_MAP = 1100` (approximate Nova Scotia values)

### K-means / medoid extraction slow or fails (Cell 6)
- If `n_aoi_pixels` > 500k, increase `AOI_GRID_SCALE` to 60 m
- If `get_cluster_medoid()` times out, reduce `N_CLUSTERS` or `AOI_GRID_SCALE`
- `medoids_data` is plain Python — safe to save: `import json; json.dump(medoids_data, open('medoids_raw.json','w'))`

### GeoTIFF export to Google Drive fails (Cell 7)
- Check `export_task.status()` for `error_message`
- Try `scale=60` if the AOI is large
- The 3 CSV files are downloaded immediately regardless of GeoTIFF status

### R model usage — embedding similarity weights
```r
# Run 1: Wadoux covariate-based weights
model_cov <- ranger(..., case.weights = df$wadoux_weight)

# Run 2: Embedding similarity weights
model_emb <- ranger(..., case.weights = df$embedding_max_sim)
```

### Expected output files
| File | Rows | Cols |
|------|------|------|
| `CorePoints_Covariates_CSV.csv` | ~8,000–13,000 | ~25 |
| `CorePoints_EmbeddingSimilarity_CSV.csv` | ~8,000–13,000 | 4 + k+2 sim cols |
| `AOI_Medoids.csv` | k=5 | 65 (cluster_id + 64 bands) |
| `AOI_KMeans_Clusters.tif` | — | 2 bands (cluster_id, sim) |
| `CorePoints_Climate_Filtered_OUT.csv` | removed profiles | all metadata + MAT/MAP |